# The illusion of a good model: row-level splits vs. leave-cell-out

This is the credibility argument for this whole library. The first version of this project's
SOH model reported R2=0.96. That number came from a row-level train/test split on a
concatenated multi-cell dataset. It was **not honest** — and this notebook reproduces exactly
why, then shows the number that actually is honest.

And even the honest number alone doesn't say how much of it the model is *earning* --
SOH-vs-cycle curves are smooth, so this notebook also checks the honest LCO R2 against a
trivial linear-fit baseline under the same fold structure.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score

from batlab.datasets import load_nasa_cells
from batlab.features import build_features, get_model_matrix
from batlab.models import GBRT_PARAMS
from batlab.validation import run_lco

cells = load_nasa_cells()
featured = {cid: build_features(df) for cid, df in cells.items()}
print(f"{len(cells)} NASA cells: {sorted(cells)}")

  [cache] B0005 already parsed (168 cycles)
  [cache] B0006 already parsed (168 cycles)
  [cache] B0007 already parsed (168 cycles)
  [cache] B0018 already parsed (132 cycles)
4 NASA cells: ['B0005', 'B0006', 'B0007', 'B0018']


## The naive approach: concatenate everything, split randomly

This is what a first attempt at "train/test split" usually looks like: put all cells' cycles
in one big table, shuffle, hold out 20%.

In [2]:
X_concat = pd.concat([get_model_matrix(f)[0] for f in featured.values()])
y_concat = pd.concat([get_model_matrix(f)[1] for f in featured.values()])

X_train, X_test, y_train, y_test = train_test_split(
    X_concat, y_concat, test_size=0.2, shuffle=True, random_state=42
)

naive_model = GradientBoostingRegressor(**GBRT_PARAMS).fit(X_train, y_train)
naive_r2 = r2_score(y_test, naive_model.predict(X_test))
print(f"Naive row-level-split SOH R2: {naive_r2:.3f}")

Naive row-level-split SOH R2: 0.998


## Why that number is a lie

`test_size=0.2, shuffle=True` on the *concatenated* table doesn't hold out a cell — it holds
out random *cycles*, scattered across all 4 cells. The model has already seen the other 99%
of every one of those cells' cycles during training, including cycles right next to the held-out
ones. It isn't being asked "can you generalize to a cell you've never seen?" — it's being asked
"can you interpolate between two cycles you've already seen the neighbors of?" That's a much
easier, and much less useful, question.

## The honest approach: leave-cell-out

Hold out an entire cell — train on the other 3, test on the 4th, never seen during training.

In [3]:
lco = run_lco(cells)
print(f"Leave-cell-out SOH R2:  {lco['soh_r2']:.3f}")
print(f"Leave-cell-out RUL R2:  {lco['rul_r2']:.3f}")
print()
print(f"Naive (row-level split):  R2 = {naive_r2:.3f}")
print(f"Honest (leave-cell-out):  R2 = {lco['soh_r2']:.3f}")
print(f"Gap: {naive_r2 - lco['soh_r2']:.3f}")

Leave-cell-out SOH R2:  0.806
Leave-cell-out RUL R2:  0.797

Naive (row-level split):  R2 = 0.998
Honest (leave-cell-out):  R2 = 0.806
Gap: 0.192


## Is the model earning its accuracy, or is this just smooth curves?

The honest LCO number above is real, but on its own it still doesn't say how much of it the
GradientBoostingRegressor and its 15+ engineered features are actually earning. Li-ion SOH-vs-cycle
curves are smooth and mostly monotonic — a **trivial linear fit against cycle_number alone**, with
no feature engineering at all, might already explain most of the variance. If it does, the honest
R2 above is still honest, but the *model* isn't where the credit belongs.

Same leave-cell-out fold structure as run_lco() (each cell held out once, trained on the other 3)
so this is an apples-to-apples comparison, not a different, easier evaluation.

In [4]:
from sklearn.linear_model import LinearRegression

baseline_r2s = []
for test_cell in cells:
    train_cells = [c for c in cells if c != test_cell]

    train_df = pd.concat([cells[c][["cycle_number", "soh_pct"]] for c in train_cells])
    test_df  = cells[test_cell][["cycle_number", "soh_pct"]]

    baseline = LinearRegression().fit(train_df[["cycle_number"]], train_df["soh_pct"])
    baseline_pred = baseline.predict(test_df[["cycle_number"]])
    baseline_r2s.append(r2_score(test_df["soh_pct"], baseline_pred))

baseline_r2 = sum(baseline_r2s) / len(baseline_r2s)
print(f"Trivial baseline (linear fit, cycle_number -> SOH only), leave-cell-out: R2 = {baseline_r2:.3f}")
print(f"GBRT, leave-cell-out, full engineered feature set:                      R2 = {lco['soh_r2']:.3f}")
print(f"GBRT's advantage over the trivial baseline:                             {lco['soh_r2'] - baseline_r2:+.3f}")

Trivial baseline (linear fit, cycle_number -> SOH only), leave-cell-out: R2 = 0.603
GBRT, leave-cell-out, full engineered feature set:                      R2 = 0.806
GBRT's advantage over the trivial baseline:                             +0.203


## Per-cell breakdown

The gap isn't uniform — some cells generalize better than others, and the *aggregate* honest
number can still hide a cell the model genuinely can't predict. This is why batlab surfaces a
**per-cell** reliability gate (`RUL_RELIABLE_FLOOR`), not just a dataset-average one.

In [5]:
from batlab.validation import RUL_RELIABLE_FLOOR
for cell_id, m in lco["per_cell"].items():
    reliable = m["rul_r2"] >= RUL_RELIABLE_FLOOR
    print(f"  {cell_id}: SOH R2={m['soh_r2']:.3f}  RUL R2={m['rul_r2']:.3f}  "
          f"{'reliable' if reliable else 'NOT reliable — would show Calibrating'}")

  B0005: SOH R2=0.927  RUL R2=0.910  reliable
  B0006: SOH R2=0.629  RUL R2=0.464  reliable
  B0007: SOH R2=0.973  RUL R2=0.890  reliable
  B0018: SOH R2=0.698  RUL R2=0.924  reliable


## Cite this

In [6]:
import batlab
print(batlab.cite())

@software{batlab,
  title  = {batlab: a citable, honest research library for battery degradation analysis},
  author = {Hosseini, Ali},
  year   = {2026},
  url    = {https://github.com/seyedali1996lb-svg/battery-intelligence-platform},
  doi    = {10.5281/zenodo.21346275},
  note   = {Standardized dataset loaders, leave-cell-out-validated SOH/RUL models, and reproducible benchmark manifests.}
}
